In [1]:
import torch
import torch.nn as nn
import torch.nn.functional as F

# 1. Define Teacher and Student Architectures
class TeacherNet(nn.Module):
    def __init__(self):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(784, 1200),
            nn.ReLU(),
            nn.Linear(1200, 1200),
            nn.ReLU(),
            nn.Linear(1200, 10)
        )
    def forward(self, x):
        return self.net(x)

class StudentNet(nn.Module):
    def __init__(self):
        super().__init__()
        # Significantly smaller hidden dimensions (Compression ratio ~10x)
        self.net = nn.Sequential(
            nn.Linear(784, 128),
            nn.ReLU(),
            nn.Linear(128, 10)
        )
    def forward(self, x):
        return self.net(x)

# 2. Instantiate and Setup Models
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

teacher = TeacherNet().to(device)
student = StudentNet().to(device)

# The teacher is pre-trained and frozen
teacher.eval()
for param in teacher.parameters():
    param.requires_grad = False

student.train()
optimizer = torch.optim.Adam(student.parameters(), lr=0.001)

# 3. Define Distillation Hyperparameters
temperature = 4.0   # T > 1 softens the probability distribution
alpha = 0.7         # Weight factor balancing soft vs. hard loss

# Mock Batch Data
mock_x = torch.randn(32, 784).to(device)
mock_y = torch.randint(0, 10, (32,)).to(device)

# 4. Execute Knowledge Distillation Step
optimizer.zero_grad()

# Forward pass on frozen teacher
with torch.no_grad():
    teacher_logits = teacher(mock_x)

# Forward pass on student
student_logits = student(mock_x)

# --- Loss Calculation ---
# A. Soft Loss: KL-Divergence on temperature-scaled probabilities
# F.kl_div expects log-probabilities for the input argument
student_soft_log_probs = F.log_softmax(student_logits / temperature, dim=-1)
teacher_soft_probs = F.softmax(teacher_logits / temperature, dim=-1)

soft_loss = F.kl_div(
    student_soft_log_probs, 
    teacher_soft_probs, 
    reduction="batchmean"
) * (temperature ** 2)

# B. Hard Loss: Standard Cross-Entropy against true labels (T=1)
hard_loss = F.cross_entropy(student_logits, mock_y)

# C. Combined Loss
total_loss = (alpha * soft_loss) + ((1.0 - alpha) * hard_loss)

# Backpropagation updates ONLY the student model parameters
total_loss.backward()
optimizer.step()

print("--- Distillation Step Execution Successful ---")
print(f"Soft Target KL Loss (Scaled): {soft_loss.item():.4f}")
print(f"Hard Label Cross-Entropy Loss: {hard_loss.item():.4f}")
print(f"Total Loss Objective:         {total_loss.item():.4f}")

--- Distillation Step Execution Successful ---
Soft Target KL Loss (Scaled): 0.0257
Hard Label Cross-Entropy Loss: 2.3537
Total Loss Objective:         0.7241
